# 03_text_embeddings
This notebook encodes product descriptions and queries into embeddings.


In [1]:
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

d:\DL_CP\multimodal-ecommerce-recommender\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
products = pd.read_csv(
    "../data/processed/products.csv"
)

print("Products:", len(products))
print(products.columns.tolist())

Products: 5000
['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'product_id', 'bought_together', 'subtitle', 'author']


In [3]:
def clean_text(value):

    if pd.isna(value):
        return ""

    return str(value)

In [4]:
products["title"] = products["title"].apply(clean_text)
products["features"] = products["features"].apply(clean_text)
products["description"] = products["description"].apply(clean_text)

In [5]:
products["combined_text"] = (
    products["title"]
    + " "
    + products["features"]
    + " "
    + products["description"]
)

print(
    products["combined_text"].iloc[0]
)

BALEAF Women's Long Sleeve Zip Beach Coverup UPF 50+ Sun Protection Hooded Cover Up Shirt Dress with Pockets ['90% Polyester, 10% Spandex' 'Zipper closure' 'Machine Wash'
 'Long sleeve sun protection coverups--UPF 50+ blocks the sun from burning'
 'Zipped v-neckline--fashionable V neck and smooth 1/4 zipper allows to staying place as you like'
 'Two drop-in side pockets--hold your phone or keys well，no worries of falling out'
 'Hoodie with non-slip drawcord--Enhancing hooded design is convenient to wrap your face and enough space to put your head and hair easily'
 'A flattering coverups company you spend all day on the beach，traveling with lovers or busying around house. Recommended For everyday leisure or daily exercise'] []


In [6]:
text_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Sentence-BERT loaded")

d:\DL_CP\multimodal-ecommerce-recommender\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Aksha\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
d:\DL_CP\multimodal-ecommerce-recommender\.venv\Lib\site-packages\huggingface_hub\

Sentence-BERT loaded


In [7]:
texts = products[
    "combined_text"
].tolist()

text_embeddings = text_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(
    "Text embedding shape:",
    text_embeddings.shape
)

Batches: 100%|██████████| 157/157 [03:00<00:00,  1.15s/it]

Text embedding shape: (5000, 384)


In [8]:
np.save(
    "../data/embeddings/text_embeddings.npy",
    text_embeddings
)

pd.DataFrame({
    "product_id": products["product_id"]
}).to_csv(
    "../data/embeddings/text_product_ids.csv",
    index=False
)

print("Text embeddings saved.")

Text embeddings saved.
